# Aviation Emissions EDA

European aviation pollutant data (CO₂, NOx, SOx) by state, segment, and flight phase (2019–2024). Source: EUROCONTROL.

## 1. Setup

In [ ]:
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root()
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, name: str) -> None:
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


In [ ]:
BLUE   = "#2176AE"
ORANGE = "#E76F51"
TEAL   = "#2A9D8F"
SLATE  = "#264653"
RED    = "#E63946"
GRAY   = "#C8C8C8"
YELLOW = "#E9C46A"

plt.rcParams.update({
    "figure.dpi":          130,
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           False,
    "font.family":         "sans-serif",
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      10,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "legend.frameon":      False,
    "figure.titlesize":    13,
    "figure.titleweight":  "bold",
})

## 2. Load Clean Data

In [ ]:
df_state = pd.read_csv(CLEAN_DIR / "emission_state_clean.csv", parse_dates=["DATE"])
df_net = pd.read_csv(CLEAN_DIR / "emission_network_clean.csv", parse_dates=["DATE"])
print(f"STATE rows: {len(df_state):,}  |  NETWORK rows: {len(df_net):,}")
df_net.head()


## 3. Exploratory Data Analysis

In [ ]:
CODE2NAME = {
    "AL": "Albania",     "AM": "Armenia",     "AT": "Austria",      "AZ": "Azerbaijan",
    "BA": "Bosnia",      "BE": "Belgium",      "BG": "Bulgaria",     "CH": "Switzerland",
    "CY": "Cyprus",      "CZ": "Czechia",      "DE": "Germany",      "DK": "Denmark",
    "EE": "Estonia",     "ES": "Spain",        "FI": "Finland",      "FR": "France",
    "GB": "UK",          "GE": "Georgia",      "GR": "Greece",       "HR": "Croatia",
    "HU": "Hungary",     "IE": "Ireland",      "IL": "Israel",       "IS": "Iceland",
    "IT": "Italy",       "LT": "Lithuania",    "LU": "Luxembourg",   "LV": "Latvia",
    "MA": "Morocco",     "MD": "Moldova",      "ME": "Montenegro",   "MK": "N.Macedonia",
    "MT": "Malta",       "NL": "Netherlands",  "NO": "Norway",       "PL": "Poland",
    "PT - Lisbon FIR":   "Portugal",
    "PT - Santa Maria FIR": "PT Azores",
    "RO": "Romania",     "RS": "Serbia",       "SE": "Sweden",       "SI": "Slovenia",
    "SK": "Slovakia",    "TR": "Turkey",       "UA": "Ukraine",
}

### 3.1 ECAC Total Emissions Over Time

In [ ]:
ecac = df_net[df_net["AREA"] == "ECAC"].copy()
ecac_monthly = (
    ecac.groupby("DATE")[["NB_FLIGHTS", "CO2_KG", "NOX_KG", "SOX_KG"]]
    .sum()
    .reset_index()
    .sort_values("DATE")
)

pollutants = [
    ("CO2_KG", "CO\u2082 (millions kg)", RED),
    ("NOX_KG", "NOx (millions kg)",      BLUE),
    ("SOX_KG", "SOx (millions kg)",      TEAL),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, (col, label, color) in zip(axes, pollutants):
    ax.fill_between(ecac_monthly["DATE"], ecac_monthly[col] / 1e6,
                    alpha=0.12, color=color)
    ax.plot(ecac_monthly["DATE"], ecac_monthly[col] / 1e6, color=color, lw=2)
    ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-01"),
               alpha=0.08, color=GRAY)
    ax.set_ylabel(label)

axes[0].set_title("ECAC monthly emissions (2019\u20132024)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
save_figure(fig, "emissions_ecac_monthly_pollutants")
plt.show()

### 3.2 CO\u2082 Seasonality by Year

In [ ]:
ecac_season = ecac.groupby(["YEAR", "MONTH"])["CO2_KG"].sum().reset_index()
piv_season  = ecac_season.pivot(index="MONTH", columns="YEAR", values="CO2_KG")

HIGHLIGHT  = {2019: BLUE, 2020: RED, 2024: TEAL}
MON_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(12, 5))
for yr in piv_season.columns:
    if yr in HIGHLIGHT:
        ax.plot(piv_season.index, piv_season[yr] / 1e9, marker="o", ms=4,
                color=HIGHLIGHT[yr], lw=2.0, label=str(yr), zorder=3)
    else:
        ax.plot(piv_season.index, piv_season[yr] / 1e9, marker="o", ms=2,
                color=GRAY, lw=1.0, alpha=0.5, label="_nolegend_", zorder=1)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MON_LABELS)
ax.set_ylabel("CO\u2082 (Mt)")
ax.set_title("ECAC CO\u2082 seasonality \u2014 monthly profile by year")
ax.legend(title="Highlighted years")
plt.tight_layout()
save_figure(fig, "emissions_co2_seasonality")
plt.show()

### 3.3 CO\u2082 by Market Segment

In [ ]:
seg = (
    df_net[df_net["AREA"] == "ECAC"]
    .groupby(["DATE", "MARKET_SEGMENT"])["CO2_KG"]
    .sum()
    .reset_index()
)
seg_piv = seg.pivot(index="DATE", columns="MARKET_SEGMENT", values="CO2_KG").fillna(0) / 1e9
order = seg_piv.sum().sort_values(ascending=False).index

SEG_COLORS = {
    "mainline": SLATE,
    "lowcost":  ORANGE,
    "business": TEAL,
    "cargo":    YELLOW,
    "regional": BLUE,
    "other":    GRAY,
}

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.stackplot(
    seg_piv.index,
    *[seg_piv[c] for c in order],
    labels=list(order),
    colors=[SEG_COLORS.get(c, GRAY) for c in order],
    alpha=0.88,
)
ax.set_ylabel("CO\u2082 (Mt)")
ax.set_title("ECAC CO\u2082 by market segment")
ax.legend(loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
save_figure(fig, "emissions_co2_market_segment")
plt.show()

### 3.4 CO\u2082 by Flight Phase

In [ ]:
phase = (
    df_net[df_net["AREA"] == "ECAC"]
    .groupby(["YEAR", "FLIGHT_PHASE"])["CO2_KG"]
    .sum()
    .reset_index()
)
phase_piv = phase.pivot(index="YEAR", columns="FLIGHT_PHASE", values="CO2_KG") / 1e9
phase_pct = phase_piv.div(phase_piv.sum(axis=1), axis=0) * 100

PHASE_COLORS = {
    "cruise":   SLATE,
    "climb":    ORANGE,
    "descent":  TEAL,
    "taxi-out": YELLOW,
    "taxi-in":  "#F4A261",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

phase_piv.plot(kind="bar", stacked=True, ax=axes[0],
               color=[PHASE_COLORS.get(c, GRAY) for c in phase_piv.columns],
               edgecolor="white", width=0.8, legend=True)
axes[0].set_ylabel("CO\u2082 (Mt)")
axes[0].set_title("CO\u2082 by flight phase absolute")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(fontsize=8)

phase_pct.plot(kind="bar", stacked=True, ax=axes[1],
               color=[PHASE_COLORS.get(c, GRAY) for c in phase_pct.columns],
               edgecolor="white", width=0.8, legend=False)
axes[1].set_ylabel("% CO\u2082")
axes[1].set_title("CO\u2082 by flight phase share")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
save_figure(fig, "emissions_co2_flight_phase")
plt.show()

### 3.5 Flights & CO\u2082 by Flight Type

In [ ]:
FT_LABELS  = {"D": "Domestic", "A": "Arrival", "I": "International", "O": "Overfly"}
FT_COLORS  = {"Domestic": RED, "Arrival": BLUE, "International": SLATE, "Overfly": YELLOW}

ft = df_state.groupby(["YEAR", "FLIGHT_TYPE"])[["CO2_KG", "NB_FLIGHTS"]].sum().reset_index()
ft["label"] = ft["FLIGHT_TYPE"].map(FT_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, ylabel, title in [
    (axes[0], "NB_FLIGHTS", "Flights (millions)",  "Number of flights by type"),
    (axes[1], "CO2_KG",     "Mt CO\u2082",           "CO\u2082 by flight type"),
]:
    piv      = ft.pivot(index="YEAR", columns="label", values=metric)
    divisor  = 1e9 if metric == "CO2_KG" else 1e6
    (piv / divisor).plot(
        kind="bar", ax=ax,
        color=[FT_COLORS[c] for c in piv.columns],
        edgecolor="white", width=0.8,
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, "emissions_flights_co2_flight_type")
plt.show()

### 3.6 Top 25 States by Total CO\u2082

In [ ]:
state_tot = (
    df_state.groupby("AREA")["CO2_KG"]
    .sum()
    .reset_index()
    .sort_values("CO2_KG", ascending=False)
    .head(25)
)
state_tot["name"] = state_tot["AREA"].map(CODE2NAME).fillna(state_tot["AREA"])

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(state_tot["name"], state_tot["CO2_KG"] / 1e9, color=SLATE, edgecolor="white")
ax.invert_yaxis()
ax.set_xlabel("Total CO\u2082 2019\u20132024 (Mt)")
ax.set_title("Top 25 states by aviation CO\u2082 emissions")

for bar, val in zip(bars, state_tot["CO2_KG"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val / 1e9:.1f}", va="center", fontsize=8, color="#444")

plt.tight_layout()
save_figure(fig, "emissions_top25_states")
plt.show()

### 3.7 CO\u2082 Efficiency per Flight

In [ ]:
ecac_eff = ecac.groupby("DATE")[["CO2_KG", "NB_FLIGHTS"]].sum().reset_index()
ecac_eff["co2_per_flight_kg"] = ecac_eff["CO2_KG"] / ecac_eff["NB_FLIGHTS"]
ecac_eff = ecac_eff.sort_values("DATE")

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax2.spines["right"].set_visible(True)
ax2.spines["left"].set_visible(False)
ax2.spines["top"].set_visible(False)

ax1.bar(ecac_eff["DATE"], ecac_eff["NB_FLIGHTS"] / 1e6,
        width=25, color=BLUE, alpha=0.3, label="Flights (millions)")
ax2.plot(ecac_eff["DATE"], ecac_eff["co2_per_flight_kg"],
         color=RED, lw=2.5, label="CO\u2082 / flight (kg)")

ax1.set_ylabel("Flights (millions)", color=BLUE)
ax2.set_ylabel("CO\u2082 per flight (kg)", color=RED)
ax1.set_title("CO\u2082 efficiency per flight vs total flight volume \u2014 ECAC monthly")

lines  = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc="upper left")

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
save_figure(fig, "emissions_co2_efficiency_per_flight")
plt.show()

### 3.8 Country Recovery Fairness: Flights vs CO2

This chart compares each country's 2024 recovery against its 2019 baseline in two dimensions: flight volume and CO2 emissions. The diagonal marks proportional recovery. Countries above the line emitted more CO2 than their traffic recovery alone would imply, while countries below the line recovered traffic with a lower CO2 rebound. Bubble size encodes absolute 2024 CO2 so percentage changes are not separated from climate burden.

In [ ]:
import numpy as np

state_year = (
    df_state.groupby(["AREA", "YEAR"])
    .agg(flights=("NB_FLIGHTS", "sum"), co2=("CO2_KG", "sum"))
    .reset_index()
)
wide = (
    state_year[state_year["YEAR"].isin([2019, 2024])]
    .pivot(index="AREA", columns="YEAR", values=["flights", "co2"])
    .dropna()
)
state_piv = pd.DataFrame({
    "AREA":        wide.index.astype(str),
    "flight_idx":  wide["flights"][2024] / wide["flights"][2019] * 100,
    "co2_idx":     wide["co2"][2024]     / wide["co2"][2019]     * 100,
    "co2_2024_mt": wide["co2"][2024] / 1e9,
})
state_piv["gap"] = state_piv["co2_idx"] - state_piv["flight_idx"]
state_piv["name"] = state_piv["AREA"].map(CODE2NAME).fillna(state_piv["AREA"])

# Keep the common recovery domain visible and exclude the Ukraine wartime outlier.
ps = state_piv[
    state_piv["flight_idx"].between(45, 185)
    & state_piv["co2_idx"].between(45, 185)
].copy()

fig, ax = plt.subplots(figsize=(9, 8))
LIM = (45, 185)
ax.set_xlim(*LIM)
ax.set_ylim(*LIM)

ax.fill_between([LIM[0], LIM[1]], [LIM[0], LIM[1]], LIM[1],
                color=RED, alpha=0.04, zorder=0)
ax.fill_between([LIM[0], LIM[1]], LIM[0], [LIM[0], LIM[1]],
                color=BLUE, alpha=0.04, zorder=0)

ax.axvline(100, color=GRAY, lw=0.8, alpha=0.7, zorder=1)
ax.axhline(100, color=GRAY, lw=0.8, alpha=0.7, zorder=1)
ax.text(101, 183, "2019 baseline", color="#666", fontsize=9, va="top")

ax.plot(LIM, LIM, color=SLATE, ls="--", lw=1.3, zorder=2)
ax.text(125, 129, "CO2 = traffic recovery",
        color=SLATE, fontsize=9, fontstyle="italic", rotation=40, va="bottom")

ax.text(47, 183, "Emissions outpace traffic",
        color=RED, fontsize=10, fontweight="bold", va="top")
ax.text(183, 47, "Traffic outpaces emissions",
        color=BLUE, fontsize=10, fontweight="bold", ha="right", va="bottom")

colors = np.where(ps["gap"] >= 0, RED, BLUE)
sizes = (ps["co2_2024_mt"].clip(lower=0.05) ** 0.55) * 160
ax.scatter(ps["flight_idx"], ps["co2_idx"],
           s=sizes, c=colors, alpha=0.78,
           edgecolors="white", linewidths=0.9, zorder=3)

top8 = set(ps.nlargest(8, "co2_2024_mt")["name"])
for _, r in ps.iterrows():
    bold = r["name"] in top8
    ax.annotate(r["name"], (r["flight_idx"], r["co2_idx"]),
                xytext=(0, -6), textcoords="offset points",
                ha="center", va="top",
                fontsize=8 if bold else 7,
                color=SLATE if bold else "#666",
                fontweight="bold" if bold else "normal")

ax.set_xlabel("2024 flights vs 2019 (%)")
ax.set_ylabel("2024 CO2 vs 2019 (%)")
ax.set_title("Country recovery fairness: traffic vs CO2")

plt.tight_layout()
save_figure(fig, "emissions_country_recovery_fairness")
plt.show()
